# Day 07 · 에이전트 직접 구성

**에이전트 = LLM + 도구 + 루프.** 최소 루프를 만든 뒤, **루프는 한 줄도 고치지 않고** 무엇을 갈아끼우는지만 바꿔간다.

| 랩 | 무엇을 갈아끼우나 |
|---|---|
| Lab 1~4 | 루프를 키운다 — 멀티스텝 · 메모리 · self-correction |
| Lab 5 | 도구 **백엔드** (인메모리 → 원격 MCP 서버) |
| Lab 6 | 검색 도구의 **속** (문자열 매칭 → 진짜 벡터 검색) |
| Lab 7 | 도구 **종류** (업무 도구 → 코드 도구) = **코드 에이전트** |
| Lab 8 | **모델** (Qwen → 다른 NVIDIA 모델들) |

- LLM = **NVIDIA API (Qwen)**
- 도구 백엔드 = 이 노트북 안에서 정의한 작은 MCP 서버 (자기완결)


## Lab 0 · 셋업

패키지 설치 → 도구 서버 정의 → NVIDIA LLM 연결.

In [ ]:
%pip install -q fastmcp openai sentence-transformers

In [ ]:
# 이 노트북은 '도구 백엔드'로 쓸 작은 MCP 서버를 여기서 직접 정의한다 (자기완결).
# 5강 예제 server.py를 그대로 써도 되고, 배포된 URL에 붙여도 된다 (Lab 5 참고).
from fastmcp import FastMCP, Client

mcp = FastMCP("Day07AgentTools")

DOCS = [
    {"id": "vacation", "text": "연차는 사용 3영업일 전에 사내포털에서 신청한다."},
    {"id": "remote",   "text": "재택근무는 주 2회까지 가능하며 전날 18시까지 공유한다."},
    {"id": "expense",  "text": "비용 정산은 영수증을 첨부해 정산 메뉴에서 접수한다."},
]
TASKS = []
LEVELS = {"낮음", "보통", "높음"}

@mcp.tool
def search_docs(query: str, k: int = 2) -> list:
    """질문과 관련된 사내 문서를 찾는다 (읽기)"""
    scored = sorted(DOCS, key=lambda d: -sum(w in d["text"] for w in query.split()))
    return scored[:k]

@mcp.tool
def add_task(title: str) -> dict:
    """할 일을 추가한다 (부작용)"""
    TASKS.append({"id": len(TASKS) + 1, "title": title, "level": "보통"})
    return TASKS[-1]

@mcp.tool
def set_priority(task_id: int, level: str) -> dict:
    """작업 우선순위를 바꾼다. level은 낮음·보통·높음 중 하나 (아니면 친절한 에러)."""
    if level not in LEVELS:
        raise ValueError(f"level은 {sorted(LEVELS)} 중 하나여야 합니다. 받은 값: '{level}'")
    for t in TASKS:
        if t["id"] == task_id:
            t["level"] = level
            return t
    raise ValueError(f"{task_id}번 작업이 없습니다.")

@mcp.tool
def list_tasks() -> list:
    """할 일 목록을 읽는다 (읽기)"""
    return TASKS

print("도구 서버 준비됨: search_docs · add_task · set_priority · list_tasks")

In [ ]:
# NVIDIA API로 Qwen(LLM)을 부른다. 토큰은 입력창/환경변수로 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)

_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:                     # thinking 모델은 제외 (토큰 폭주 방지)
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank", "thinking"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("모델:", LLM_MODEL)

## Lab 1 · 최소 에이전트 해부

에이전트 루프의 뼈대: **① LLM 판단 → ② 도구 실행 → ③ 결과 되먹임 → 반복**. 도구를 더 안 부르면 정상 종료, `max_steps`를 넘으면 안전 종료.

In [ ]:
# ① MCP 도구(name·description·inputSchema) → OpenAI function 스펙으로 변환
def to_openai_tools(mcp_tools):
    return [{"type": "function", "function": {
        "name": t.name, "description": t.description or "", "parameters": t.inputSchema}} for t in mcp_tools]

# ② 최소 에이전트 루프 — 판단 → 도구 실행 → 되먹임 → 반복
async def run_agent(question, max_steps=5):
    async with Client(mcp) as client:                       # 도구 백엔드 = 내 MCP 서버
        tools = to_openai_tools(await client.list_tools())
        messages = [{"role": "system", "content": "너는 업무 비서다. 필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for step in range(max_steps):                       # 반복 한도 = 안전장치
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=messages, tools=tools, temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:                            # 도구를 안 부르면 → 정상 종료
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:                         # LLM이 고른 도구를 실제 실행
                args = json.loads(tc.function.arguments)
                print(f"  [LLM → 도구] {tc.function.name}({args})")
                res = await client.call_tool(tc.function.name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(res.data, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"                            # 안전 종료

In [ ]:
# 한 번의 질문 → 에이전트가 스스로 search_docs 를 부르고, 그 근거로 답한다
print("Q: 연차는 며칠 전에 신청해?")
print("A:", await run_agent("연차는 며칠 전에 신청해?"))

## Lab 2 · 루프 승격 — 멀티스텝·종료조건

한 질문에 도구가 여러 번 필요하면 루프가 여러 번 돈다. `[LLM → 도구]` 로그가 몇 번 찍히는지 보라. `max_steps`(반복 한도)가 무한 루프·비용 폭주를 막는 안전장치다.

In [ ]:
# 멀티스텝 — 검색과 행동을 이어서 (루프가 여러 번 돈다)
print("Q: '분기 보고서 작성' 할 일을 추가하고, 지금 할 일 목록을 보여줘")
print("A:", await run_agent("'분기 보고서 작성' 할 일을 추가하고, 지금 할 일 목록을 보여줘"))

### 2-2 · 진짜 멀티스텝 — 앞 결과가 있어야 다음 도구를 부를 수 있는 과제

`"추가하고 목록 보여줘"` 는 사실 도구 두 개를 **한 번에(병렬로)** 부르면 끝난다 — 루프는 1회만 돌 수도 있다.

**진짜 멀티스텝**은 앞 도구의 **결과를 봐야** 다음 도구의 **인자를 정할 수 있는** 경우다.

> *"목록의 **마지막** 할 일의 우선순위를 높음으로"* → `list_tasks`로 **id를 알아낸 뒤**라야
> `set_priority(task_id=?)`를 부를 수 있다. 병렬로는 불가능하다.

아래 `run_agent_traced`는 **루프가 몇 번 돌았는지 세어서** 진짜 멀티스텝인지 직접 판정한다.

In [ ]:
# 루프 횟수와 도구 호출을 기록하는 추적판 run_agent (루프 본체는 동일)
async def run_agent_traced(question, max_steps=6):
    trace = []
    async with Client(mcp) as client:
        tools = to_openai_tools(await client.list_tools())
        messages = [{"role": "system", "content": "너는 업무 비서다. 필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for step in range(1, max_steps + 1):
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=messages, tools=tools, temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:
                return m.content, trace, step - 1        # 도구를 부른 '루프' 횟수
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                trace.append((step, tc.function.name, args))
                print(f"  [루프 {step}] {tc.function.name}({args})")
                res = await client.call_tool(tc.function.name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(res.data, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)", trace, max_steps


def report(trace, loops):
    """진짜 멀티스텝이었는지 스스로 판정한다"""
    print(f"\n→ 도구를 부른 루프 {loops}회 · 총 도구 호출 {len(trace)}회")
    if loops >= 2:
        print("✅ 진짜 멀티스텝 — 앞 결과를 보고 '다음' 도구를 정했다")
    elif len(trace) >= 2:
        print("⚠️ 한 루프에서 여러 도구를 '병렬' 호출 — 순차 의존이 아니다")
    else:
        print("⚠️ 도구를 한 번만 불렀다")

In [ ]:
# id를 '추측'으로 맞출 수 없도록 할 일을 미리 넣어둔다
TASKS.clear()
TASKS.extend([{"id": 1, "title": "보안 교육 이수",   "level": "보통"},
              {"id": 2, "title": "분기 보고서 초안", "level": "보통"},
              {"id": 3, "title": "채용 면접 준비",   "level": "보통"}])

print("Q: 지금 할 일 목록에서 '마지막' 항목의 우선순위를 높음으로 바꿔줘\n")
ans, trace, loops = await run_agent_traced("지금 할 일 목록에서 '마지막' 항목의 우선순위를 높음으로 바꿔줘.")
print("\nA:", ans)
report(trace, loops)
print("\n실제 결과:", TASKS)          # 3번 항목만 '높음' 이어야 성공

## Lab 2.5 · Plan-and-Execute — 계획 먼저

ReAct는 **매 스텝 즉흥으로** 다음 도구를 정한다. 복잡한 일에서는 길을 잃는다.
**Plan-and-Execute**는 **먼저 전체 계획을 세우고** 그 다음 실행한다.

핵심은 코드 한 줄이다 — **계획 단계에는 `tools=`를 넘기지 않는다.**

| | ReAct | Plan-and-Execute |
|---|---|---|
| 계획 시점 | 매 스텝 즉흥 판단 | **먼저 전체 계획** 후 실행 |
| `tools=` | 항상 넘김 | **계획 단계엔 안 넘김** ← 핵심 |
| 강점 | 유연·적응적 | 다단계에서 길을 안 잃음 |
| 약점 | 긴 작업에서 헤맴 | 계획이 틀리면 통째로 틀림 |

---

### 왜 도구를 빼야 하나 — 실제로 재봤다

같은 프롬프트(**"실행하지 말고 계획만 세워라"**)로 Qwen에 5회씩 물었다:

| 조건 | 결과 |
|---|---|
| `tools` 안 넘김 | 계획 텍스트 **5/5** ✅ |
| `tools` 넘김 | **도구 호출 5/5** — 지시를 무시하고 `add_task`를 3번 불러버렸다 |
| `tools` + `tool_choice="none"` | 🚨 **계획 0/5.** `<tool_call>` **원문이 그대로 새어나왔다** |

**`tool_choice="none"`은 해결책이 아니다.** 도구 호출이 *파싱만* 안 될 뿐, 모델은 여전히 도구를 부르려 한다.
그래서 `tool_calls`는 비어 있는데 `content`에 이런 쓰레기가 담긴다:

```
<tool_call>
{"name": "add_task", "arguments": {"title": "분기 보고서 데이터 수집"}}
</tool_call>
```

이걸 계획인 줄 알고 다음 단계에 넘기면 컨텍스트가 오염된다.
→ **계획 단계에는 `tools`를 아예 넘기지 마라.** 그게 유일하게 통하는 방법이다.

> 직접 확인: `python day07/_실험_계획단계에_도구를_주면.py`

In [ ]:
async def plan_and_execute(question, max_steps=8):
    # ① 계획 — tools 를 '안' 넘기는 게 핵심. 주면 계획 대신 그냥 실행해버린다.
    plan = llm.chat.completions.create(
        model=LLM_MODEL, temperature=0.2, max_tokens=300,
        messages=[{"role": "system", "content": "해결 단계를 번호로 나열하라. 실행하지 말고 계획만 세워라."},
                  {"role": "user", "content": question}],
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    ).choices[0].message.content
    print("[계획]\n" + plan + "\n\n[실행]")

    # ② 실행 — 계획을 컨텍스트에 얹고, 평소의 ReAct 루프를 그대로 돈다
    return await run_agent_traced(f"[계획]\n{plan}\n\n[요청] {question}", max_steps)

In [ ]:
TASKS.clear()

TASK_Q = "'분기 보고서' 관련 할 일 3개를 만들고, 그중 마지막 것만 우선순위를 높음으로 올려줘."
ans, trace, loops = await plan_and_execute(TASK_Q)

print("\nA:", ans)
report(trace, loops)
print("\n실제 결과:")
for t in TASKS:
    print("  ", t)

## Lab 3 · 메모리 — 대화가 곧 기억

`run_agent`는 매 호출이 새 대화라 이전을 기억하지 못한다. `messages`를 여러 질문에 걸쳐 유지하는 `Agent`를 만들면 "방금 그 작업" 같은 지시를 이해한다.

> 대화가 길어지면 요약·트리밍이 필요해진다 → 8강 컨텍스트 엔지니어링.

In [ ]:
# 메모리 = 대화(messages)를 여러 질문에 걸쳐 유지하는 에이전트
class Agent:
    def __init__(self, system):
        self.system = system
        self.messages = [{"role": "system", "content": system}]

    async def ask(self, question, max_steps=5):
        async with Client(mcp) as client:
            tools = to_openai_tools(await client.list_tools())
            self.messages.append({"role": "user", "content": question})   # 이전 대화 위에 쌓인다
            for step in range(max_steps):
                m = llm.chat.completions.create(
                    model=LLM_MODEL, messages=self.messages, tools=tools, temperature=0.2, max_tokens=400,
                    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
                ).choices[0].message
                if not m.tool_calls:
                    self.messages.append({"role": "assistant", "content": m.content})
                    return m.content
                self.messages.append({"role": "assistant", "content": m.content or "",
                                      "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
                for tc in m.tool_calls:
                    args = json.loads(tc.function.arguments)
                    print(f"  [LLM → 도구] {tc.function.name}({args})")
                    try:
                        res = await client.call_tool(tc.function.name, args)
                        content = json.dumps(res.data, ensure_ascii=False, default=str)
                    except Exception as e:              # 에러도 관찰로 되먹임 → 스스로 수정
                        content = f"오류: {str(e).splitlines()[-1]}"
                        print(f"       ↳ 오류를 LLM에 전달: {content}")
                    self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
            return "(반복 한도 초과)"

In [ ]:
# 메모리 확인 — 두 번째 질문의 '방금 그 작업'은 이전 대화를 기억해야 풀린다
agent = Agent("너는 업무 비서다. 도구로 처리하고 한국어로 간결히 답하라.")
print("Q1:", "'디자인 리뷰' 할 일을 추가해줘")
print("A1:", await agent.ask("'디자인 리뷰' 할 일을 추가해줘"))
print("\nQ2:", "방금 추가한 그 작업을 우선순위 높음으로 바꿔줘")
print("A2:", await agent.ask("방금 추가한 그 작업을 우선순위 높음으로 바꿔줘"))

## Lab 4 · self-correction — 에러를 되먹여 스스로 고친다

도구가 던진 에러를 **관찰처럼 되먹이면** LLM이 인자를 고쳐 다시 시도한다. 그래서 5강에서 배운 **친절한 에러**가 중요하다 — 뭘 어떻게 고칠지 알려줘야 한다.

In [ ]:
# self-correction — '긴급'은 허용값(낮음·보통·높음)이 아니라 set_priority 가 친절한 에러를 던진다.
# 에이전트는 그 에러를 읽고 '높음'으로 고쳐 다시 부른다 ([LLM → 도구] 로그가 두 번 찍힌다).
agent2 = Agent("너는 업무 비서다. 도구로 처리하고 한국어로 간결히 답하라.")
print("Q:", "'발표 준비' 할 일을 추가하고, 그 작업을 '긴급'으로 설정해줘")
print("A:", await agent2.ask("'발표 준비' 할 일을 추가하고, 그 작업을 '긴급'으로 설정해줘"))

## Lab 5 · 도구 백엔드 = 진짜 MCP 서버

'에이전트 루프'와 '도구가 어디 있는지'는 분리돼 있다. `Client(mcp)`(인메모리)를 `Client("http://...")`(원격/배포)로 바꿔도 루프는 그대로다.

In [ ]:
# Lab 5 — 도구 백엔드는 '진짜 MCP 서버'다. 인메모리(Client(mcp)) 대신
# 배포된 서버 URL이나 로컬 HTTP로 바꿔도 run_agent 는 그대로 동작한다.
#
# 예) 5강 deploy 서버를 로컬 HTTP로 띄운 뒤:
#     터미널> fastmcp run day05/deploy/server.py --transport http --port 8000
#
# async def run_agent_remote(question, url="http://localhost:8000/mcp", max_steps=5):
#     async with Client(url) as client:          # ← Client(mcp) 를 Client(url) 로만 바꿈
#         tools = to_openai_tools(await client.list_tools())
#         ...                                      # 나머지 루프는 동일
#
# 즉 '에이전트 루프'와 '도구가 어디 있는지'는 분리돼 있다.
print("도구 백엔드는 Client(mcp)=인메모리 ↔ Client(url)=원격 으로 바꿔 끼울 수 있다.")

## Lab 6 · 검색 도구를 진짜 RAG로 갈아끼우기

Lab 0의 `search_docs`는 **단어가 겹치는 개수를 세는 장난감**이었다. 이제 Day 1~2에서 배운 **벡터 검색**으로 속을 바꾼다.

바뀌는 건 도구의 **속**뿐이다 — 도구 이름도, 시그니처도, `run_agent` 루프도 그대로다.

> Day 02 노트북은 이렇게 끝났다: *"이 검색기가 Week 3에서 **에이전트의 검색 도구**가 됩니다."* — 그 약속을 여기서 지킨다.

In [ ]:
# Day 1~2의 임베딩 + 벡터 검색. 모델은 Day01에서 이미 받아놔 캐시에 있다(첫 실행은 다운로드로 좀 걸릴 수 있음).
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")   # 로컬·CPU

# 장난감 3문장 대신 조금 더 현실적인 사내 문서
DOCS = [
    {"id": "vacation", "text": "연차 휴가는 사용 3영업일 전까지 사내포털에서 신청하고 팀장 승인을 받는다."},
    {"id": "remote",   "text": "재택근무는 주 2회까지 가능하며 전날 18시까지 팀 채널에 공유한다."},
    {"id": "expense",  "text": "비용 정산은 영수증을 첨부해 신청하고, 경영지원팀 검토 후 재무팀이 최종 승인한다."},
    {"id": "security", "text": "외부 USB 사용은 금지이며 파일 공유는 사내 승인 드라이브만 사용한다."},
    {"id": "err4021",  "text": "에러코드 ERR-4021은 인증 토큰 만료를 의미하며 재로그인으로 해결한다."},
    {"id": "err5010",  "text": "에러코드 ERR-5010은 네트워크 연결 문제이며 VPN 상태를 먼저 확인한다."},
    {"id": "onboard",  "text": "신입사원은 입사 첫날 보안 서약서를 작성하고 1주일 내 정보보안 교육을 이수한다."},
    {"id": "firmware", "text": "제품 X-200의 펌웨어 최신 버전은 2.3이며 설정 메뉴에서 업데이트한다."},
]
DOC_VECS = embedder.encode([d["text"] for d in DOCS], normalize_embeddings=True)
print("문서", len(DOCS), "개 임베딩 완료 →", DOC_VECS.shape)

In [ ]:
# 같은 이름으로 도구를 다시 등록한다 — 겉(이름·인자)은 그대로, 속만 벡터 검색으로.
mcp.remove_tool("search_docs")            # 장난감 도구를 뗀다

@mcp.tool
def search_docs(query: str, k: int = 3) -> list:
    """질문과 의미가 가까운 사내 문서를 찾는다 (벡터 검색)"""
    qv = embedder.encode([query], normalize_embeddings=True)[0]
    sims = DOC_VECS @ qv                   # 정규화된 벡터 → 내적 = 코사인
    top = np.argsort(-sims)[:k]
    return [{**DOCS[i], "score": round(float(sims[i]), 3)} for i in top]

print("search_docs → 벡터 검색으로 교체됨 (루프는 한 줄도 안 바뀜)")

### Agentic RAG — 검색할지 말지를 **에이전트가** 정한다

Day 1~2에서는 질문이 오면 검색이 **항상** 돌았다. 이제는 다르다. 도구를 쥐여줬을 뿐,
**부를지 말지는 LLM이 판단**한다. `[LLM → 도구]` 로그가 찍히는지 보라.

In [ ]:
# 검색이 필요 없는 질문 → 도구를 안 부른다 (로그가 안 찍힌다)
print("Q1: 2 더하기 2는?")
print("A1:", await run_agent("2 더하기 2는?"))

# 사내 규정을 물으면 → 스스로 search_docs 를 부른다
print("\nQ2: 로그인이 자꾸 풀리는데 왜 그런가요?")   # 'ERR-4021'을 직접 말하지 않았다
print("A2:", await run_agent("로그인이 자꾸 풀리는데 왜 그런가요?"))

### 검색 → 행동 — 도구를 **가로질러** 이어지는 멀티스텝

이제 에이전트는 **검색 도구(RAG)** 와 **행동 도구**를 둘 다 갖고 있다.

> *"네트워크가 안 될 때 뭘 확인해야 하는지 **찾아보고**, 그 조치를 **할 일로 등록**해줘"*

`add_task`의 **제목**은 `search_docs`가 무엇을 찾아왔는지 **본 뒤에야** 정할 수 있다.
종류가 다른 도구를 가로질러 순차 의존이 생기는, 가장 실전에 가까운 멀티스텝이다.

In [ ]:
TASKS.clear()

Q = "네트워크가 안 될 때 뭘 확인해야 하는지 문서에서 찾아보고, 그 조치를 할 일로 등록해줘."
print("Q:", Q, "\n")

ans, trace, loops = await run_agent_traced(Q)

print("\nA:", ans)
report(trace, loops)
print("\n등록된 할 일:", TASKS)      # 검색 결과(VPN 확인)에서 나온 제목이어야 한다

### ⚠️ 벡터 검색의 한계 — 정확 토큰에 약하다

위 질문은 **의미형**("네트워크가 안 될 때")이라 잘 찾았다. 그런데 **에러코드를 직접** 물으면 어떻게 될까?

`ERR-4021`과 `ERR-5010`은 문장 구조가 거의 같아서, 임베딩이 **숫자를 구분하지 못한다.**
직접 확인해보라 — 이건 Day 02에서 배운 그대로다.

In [ ]:
for q in ["ERR-5010", "네트워크가 안 될 때 확인할 것"]:
    qv = embedder.encode([q], normalize_embeddings=True)[0]
    sims = DOC_VECS @ qv
    print(f"[{q}]")
    for i in np.argsort(-sims)[:3]:
        star = "  ← 정답" if ("5010" in q and DOCS[i]["id"] == "err5010") else ""
        print(f"   {float(sims[i]):.3f}  {DOCS[i]['id']:<9} {DOCS[i]['text'][:30]}{star}")
    print()

print("'ERR-5010' 을 물었는데 ERR-4021 이 1등으로 나온다 — 두 문장이 거의 같아")
print("임베딩이 숫자를 구분하지 못한다. 정확 토큰에는 BM25(키워드) 가 필요하다.")
print("→ Day 02의 하이브리드 검색(BM25 + 벡터)이 존재하는 이유다.")

## Lab 7 · 미니 코드 에이전트

루프는 그대로 두고 **도구만 코드 도구로** 바꾼다 — `list_files` · `read_file` · `write_file` · `run_python`.

그러면 **코드 에이전트**가 된다. 프로덕션 코드 에이전트의 뼈대가 바로 이것이다.

세 단계로 확인한다:
1. **버그 수정** — 실패하는 테스트를 스스로 통과시킨다
2. **`run_python`을 빼면?** — 고쳤다고 답하지만 **검증을 못 한다**


In [ ]:
# 코드 에이전트가 놀 샌드박스. 이 폴더 밖으로는 못 나간다.
import sys, subprocess, shutil
from pathlib import Path

SANDBOX = Path("sandbox").resolve()

BUGGY = '''def apply_discount(price, percent):
    """percent% 할인가를 돌려준다."""
    return price - percent        # 버그: 퍼센트를 '금액'처럼 빼고 있다
'''

# 요구사항은 '테스트'에 적혀 있다 — 할인가는 정수여야 한다.
# 코드만 읽어서는 알 수 없고, 실행해봐야 걸린다.
TEST = '''from calc import apply_discount

got = apply_discount(10000, 10)
assert got == 9000, f"10000원의 10% 할인가는 9000이어야 하는데 {got}"
assert isinstance(got, int), f"할인가는 정수여야 하는데 {type(got).__name__}({got})가 나왔다"

got = apply_discount(20000, 25)
assert got == 15000, f"20000원의 25% 할인가는 15000이어야 하는데 {got}"
assert isinstance(got, int), f"할인가는 정수여야 하는데 {type(got).__name__}({got})가 나왔다"

print("테스트 통과")
'''

def reset_sandbox():
    """샌드박스를 '버그가 있는 처음 상태'로 되돌린다"""
    if SANDBOX.exists():
        shutil.rmtree(SANDBOX)
    SANDBOX.mkdir()
    (SANDBOX / "calc.py").write_text(BUGGY, encoding="utf-8")
    (SANDBOX / "test_calc.py").write_text(TEST, encoding="utf-8")

def test_passes() -> bool:
    """우리가 직접 테스트를 돌려 '진짜' 고쳐졌는지 확인한다 (채점용)"""
    r = subprocess.run([sys.executable, "test_calc.py"], cwd=SANDBOX,
                       capture_output=True, text=True, timeout=20)
    return r.returncode == 0

reset_sandbox()
print("샌드박스 준비:", sorted(p.name for p in SANDBOX.iterdir()))
print("지금 테스트는 통과하나? →", test_passes(), "  (버그가 있으니 False 가 정상)")

In [ ]:
# 코드 도구 4개 — 전부 샌드박스 안으로만 제한된다.
code_mcp = FastMCP("CodeAgentTools")

def _safe(path: str) -> Path:
    p = (SANDBOX / path).resolve()
    if not p.is_relative_to(SANDBOX):                 # 샌드박스 탈출 차단
        raise ValueError(f"샌드박스({SANDBOX.name}/) 밖은 접근 금지: {path}")
    return p

@code_mcp.tool
def list_files() -> list:
    """작업 폴더의 파일 목록을 본다"""
    return sorted(p.name for p in SANDBOX.iterdir() if p.is_file())

@code_mcp.tool
def read_file(path: str) -> str:
    """파일 내용을 읽는다"""
    return _safe(path).read_text(encoding="utf-8")

@code_mcp.tool
def write_file(path: str, content: str) -> str:
    """파일에 내용을 쓴다 (파일을 통째로 덮어쓴다)"""
    _safe(path).write_text(content, encoding="utf-8")
    return f"{path} 저장 완료 ({len(content)}자)"

@code_mcp.tool
def run_python(path: str) -> str:
    """파이썬 파일을 실행하고 결과(stdout/stderr)를 돌려준다"""
    r = subprocess.run([sys.executable, str(_safe(path))], cwd=SANDBOX,
                       capture_output=True, text=True, timeout=20)
    return f"exit={r.returncode}\n[stdout]\n{r.stdout}\n[stderr]\n{r.stderr}"

print("코드 도구 4개 등록:", "list_files · read_file · write_file · run_python")

In [ ]:
# Lab 1의 run_agent 와 '똑같은 루프'다. 서버·시스템 프롬프트·허용 도구만 인자로 뺐다.
async def run_loop(server, system, question, max_steps=8, allow=None, verbose=True):
    async with Client(server) as client:
        tools = await client.list_tools()
        if allow is not None:
            tools = [t for t in tools if t.name in allow]      # 도구를 골라서 준다
        specs = to_openai_tools(tools)
        messages = [{"role": "system", "content": system},
                    {"role": "user", "content": question}]
        for step in range(max_steps):
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=messages, tools=specs, temperature=0.2, max_tokens=1200,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:
                args = json.loads(tc.function.arguments)
                if verbose:
                    peek = {k: (v[:35] + "…" if isinstance(v, str) and len(v) > 35 else v)
                            for k, v in args.items()}
                    print(f"  [LLM → 도구] {tc.function.name}({peek})")
                try:
                    res = await client.call_tool(tc.function.name, args)
                    content = json.dumps(res.data, ensure_ascii=False, default=str)
                except Exception as e:                        # 에러도 관찰로 되먹인다
                    content = f"오류: {str(e).splitlines()[-1]}"
                    if verbose:
                        print(f"       ↳ 오류 되먹임: {content}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
        return "(반복 한도 초과)"

CODER = ("너는 코드 에이전트다. 도구로 파일을 읽고·고치고·실행해서 문제를 해결하라. "
         "추측하지 말고 반드시 실행해서 확인하라. 끝나면 무엇을 고쳤는지 한국어로 간결히 답하라.")

TASK = "test_calc.py 를 실행해보고, 실패하면 원인을 찾아 고쳐서 통과시켜라."
print("준비 완료 — 루프는 Lab 1과 동일, 도구만 코드 도구다")

### 7-1 · 버그 수정 (RED → GREEN)

도구 4개를 전부 준다. 읽고 → 돌리고 → 실패를 보고 → 고치고 → 다시 돌린다.

In [ ]:
reset_sandbox()
print("[시작] 테스트 통과?", test_passes(), "\n")

answer = await run_loop(code_mcp, CODER, TASK)

print("\n에이전트:", answer)
print("[채점] 테스트 통과?", test_passes())          # True 면 진짜 고친 것
print("\n--- 고쳐진 calc.py ---")
print((SANDBOX / "calc.py").read_text(encoding="utf-8"))

### 7-2 · `run_python`을 빼면 무슨 일이 생기나 ⭐

**같은 과제 · 같은 루프 · 같은 모델.** 실행 도구 하나만 뺀다.

에이전트는 코드를 고치고 자신 있게 "완료했습니다"라고 답할 것이다. 하지만 **스스로 확인할 방법이 없다.**
우리가 대신 채점해보자.

- **실패했다면** → 에이전트는 그걸 **모른 채** '완료'라고 답한 것이다.
- **통과했다면** → **운이 좋았을 뿐이다.** 확인할 수단이 없었던 건 똑같다.

둘 다 같은 문제다 — **루프가 닫히지 않았다.** 실행(검증) 도구가 있어야 에이전트는 자기 일의 결과를 본다.

> 이게 챗봇과 코드 에이전트를 가르는 선이다.

In [ ]:
reset_sandbox()

answer = await run_loop(code_mcp, CODER, TASK,
                        allow={"list_files", "read_file", "write_file"})   # run_python 없음

print("\n에이전트:", answer)
print("[채점] 테스트 통과?", test_passes())   # 에이전트는 모른다 — 확인할 도구가 없었으니까
print("\n--- 에이전트가 쓴 calc.py ---")
print((SANDBOX / "calc.py").read_text(encoding="utf-8"))

### 7-3 · 샌드박스 경계

`_safe()` 한 줄이 에이전트를 폴더 안에 가둔다. 밖으로 나가려 하면 **에러가 되먹여지고**, 에이전트는 포기한다.



In [ ]:
print(await run_loop(code_mcp, CODER,
                     "상위 폴더에 있는 ../Day07_에이전트_실습.ipynb 파일을 읽어서 요약해줘.",
                     max_steps=3))

## Lab 8 · 모델 스위칭 벤치

**루프도 과제도 도구도 그대로.** `LLM_MODEL` 하나만 바꿔가며 같은 버그를 고치게 한다.

보게 될 것:
1. **루프는 모델과 무관하다** — 같은 코드가 전부 돌아간다
2. 작은 모델은 도구를 엉뚱하게 부르거나 루프를 못 끝내고 `max_steps`에 걸린다
3. **코드 에이전트에선 모델 선택이 곧 성능이다**

> tool-calling을 지원하지 않는 모델은 에러가 난다. 그것도 결과다 — 표에 그대로 남긴다.

In [ ]:
import time

CANDIDATES = [
    "qwen/qwen3-next-80b-a3b-instruct",   # 기본 (활성 파라미터가 작다)
    "qwen/qwen3.5-397b-a17b",             # 같은 계열 큰 모델 → 크기 효과
    "deepseek-ai/deepseek-v4-pro",        # 코딩 특화
    "openai/gpt-oss-120b",                # 오픈 GPT
    "moonshotai/kimi-k2.6",               # 에이전틱/툴콜 강함
]
CANDIDATES = [m for m in CANDIDATES if m in _av]      # 이 계정에서 쓸 수 있는 것만
print("벤치 대상:", len(CANDIDATES), "개\n")

rows = []
for model in CANDIDATES:
    _prev, LLM_MODEL = LLM_MODEL, model               # 모델만 바꾼다
    reset_sandbox()
    t0 = time.time()
    try:
        await run_loop(code_mcp, CODER, TASK, max_steps=8, verbose=False)
        result = "성공" if test_passes() else "실패"
    except Exception as e:
        result = f"에러({type(e).__name__})"                # 툴콜 미지원 등
    finally:
        took, LLM_MODEL = time.time() - t0, _prev        # 원래 모델로 복구
    rows.append((model, result, f"{took:.1f}s"))
    print(f"  {result:<12} {took:5.1f}s  {model}")

print("\n| 모델 | 결과 | 시간 |")
print("|---|---|---|")
for m, r, t in rows:
    print(f"| {m} | {r} | {t} |")

## 산출물 체크리스트

- [ ] 최소 에이전트 루프를 이해하고 돌렸다 (`run_agent`)
- [ ] 멀티스텝·종료조건·안전장치를 확인했다
- [ ] 메모리를 유지하는 `Agent`로 이어지는 지시를 처리했다
- [ ] self-correction(에러 되먹임)으로 스스로 고치는 걸 봤다
- [ ] 도구 백엔드가 MCP 서버임을 확인했다 (URL 교체 가능)
- [ ] **검색 도구의 속을 진짜 벡터 검색으로 갈아끼웠다** (Lab 6)
- [ ] **에이전트가 검색할지 말지 스스로 판단하는 걸 봤다** (Agentic RAG)
- [ ] **코드 도구를 물려 미니 코드 에이전트를 만들었다** (Lab 7)
- [ ] **`run_python`을 빼면 '검증 못 하는 에이전트'가 되는 걸 확인했다**
- [ ] **모델만 바꿔가며 같은 과제를 비교했다** (Lab 8)

